# Phase 2: Full Training
YOLO26n vs YOLO11n on VisDrone, no `fraction` cap

**Input needed:** none. If you're continuing the same Kaggle session as Phase 1, the dataset is already on disk. If this is a fresh session, the training cell below auto-downloads VisDrone again (about 40 seconds), no need to attach anything through Add Input.

**Before running:** GPU accelerator on, Internet on (only matters if the dataset isn't already cached from Phase 1).

**If your session disconnects mid-run:** rerun the training cell with `resume=True` added to the affected model's `train()` call and `model="runs/phase2_full/<name>/weights/last.pt"` instead of the `.pt` base weights, it picks up from the last checkpoint instead of starting over.

**If you're worried about the 12-hour session limit:** edit the model list in the training cell to one name at a time and run it as two separate sessions instead of one loop.

In [1]:
!pip install -q ultralytics onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.0 MB/s eta 0:00:00


In [2]:
"""Same environment check as Phase 1, kept here so this notebook is self-contained."""
import torch
import ultralytics


def check_environment() -> None:
    print(f"ultralytics version: {ultralytics.__version__}")
    print(f"torch version: {torch.__version__}")
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU detected. Enable a GPU accelerator in Kaggle notebook settings.")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


check_environment()

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics version: 8.4.124
torch version: 2.10.0+cu128
GPU: Tesla T4


In [3]:
"""
Full training run, both models, same epoch budget, no fraction cap.
optimizer is left on 'auto' for both, so the only difference between the two
runs is model architecture, not training recipe, that's what keeps the
comparison fair.
"""
import time
from ultralytics import YOLO

EPOCHS = 30
IMGSZ = 640


def train_full(model_name: str, data_yaml: str = "VisDrone.yaml") -> float:
    """Trains one model to completion, returns elapsed minutes."""
    start = time.perf_counter()
    try:
        model = YOLO(model_name)
        model.train(
            data=data_yaml,
            epochs=EPOCHS,
            imgsz=IMGSZ,
            device= 0,
            project="runs/phase2_full",
            name=model_name.replace(".pt", ""),
            verbose=True,
        )
    except Exception as e:
        raise RuntimeError(f"{model_name} full training failed: {e}") from e

    elapsed_min = (time.perf_counter() - start) / 60
    print(f"{model_name}: training complete, {elapsed_min:.1f} minutes.")
    return elapsed_min


for name in ["yolo26n.pt", "yolo11n.pt"]:
    train_full(name)

Ultralytics 8.4.124 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n, nbs=64, nms=False, opset=None, optimiz

In [4]:
"""
Pull the final epoch's metrics from both runs for a quick side-by-side.

Ultralytics resolves `project=` relative to its own runs directory and
inserts the task name in between, so the exact save path isn't always what
you passed in. This searches for it instead of hardcoding it.
"""
import pandas as pd
from pathlib import Path


def find_results_csv(model_dir_name: str, search_root: str = ".") -> Path:
    """Finds results.csv under any folder ending in model_dir_name, most recently modified first."""
    matches = sorted(
        Path(search_root).rglob(f"{model_dir_name}/results.csv"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not matches:
        raise FileNotFoundError(
            f"No results.csv found under a '{model_dir_name}' folder anywhere under {search_root}, "
            "training may not have finished."
        )
    return matches[0]


def last_epoch_metrics(model_dir_name: str) -> dict:
    csv_path = find_results_csv(model_dir_name)
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]
    return {
        "path": str(csv_path),
        "mAP50": float(last.get("metrics/mAP50(B)", float("nan"))),
        "mAP50-95": float(last.get("metrics/mAP50-95(B)", float("nan"))),
        "precision": float(last.get("metrics/precision(B)", float("nan"))),
        "recall": float(last.get("metrics/recall(B)", float("nan"))),
    }


for name, model_dir_name in [("YOLO26n", "yolo26n"), ("YOLO11n", "yolo11n")]:
    metrics = last_epoch_metrics(model_dir_name)
    print(f"{name} ({metrics['path']}): mAP50={metrics['mAP50']:.4f}  mAP50-95={metrics['mAP50-95']:.4f}  "
          f"P={metrics['precision']:.4f}  R={metrics['recall']:.4f}")

YOLO26n (runs/detect/runs/phase2_full/yolo26n/results.csv): mAP50=0.2634  mAP50-95=0.1449  P=0.3836  R=0.3028
YOLO11n (runs/detect/runs/phase2_full/yolo11n/results.csv): mAP50=0.2800  mAP50-95=0.1576  P=0.3984  R=0.3122


## Phase 2 is complete when:
- Both training runs print "training complete"
- The metrics cell prints real (non-NaN) numbers for both models

[verify] the exact `results.csv` column names against your installed ultralytics version if the metrics cell errors on a missing key, older versions used slightly different column labels.

**Next: Phase 3**, export both to ONNX and benchmark CPU latency.